# Analog Holidays - 38h Offset Forecast + Cluster Filter

Thin orchestration notebook for running `AnalogSpecialDays` from the hourly wide holiday audit CSV.

This variant forecasts a 38-hour window that starts 14 hours before the holiday midnight, so the operational forecast is ready before the holiday begins.

The loading, normalization, forecasting, and plotting logic lives in `analog/analog_holidays.py`. This notebook only defines parameters and calls the plotting helpers.

The CSV export only preserves holiday flags, so this workflow targets holiday analogs and restricts the analog bank to the selector cluster `F/G/H` assigned to each target in `holiday_selector_features.csv`.

In [ ]:
from pathlib import Path
import importlib
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import analog_holidays.analog.analog_special_days as analog_special_days_module
import analog_holidays.analog.analog_holidays as analog_holidays_module

analog_special_days_module = importlib.reload(analog_special_days_module)
analog_holidays_module = importlib.reload(analog_holidays_module)

from analog_holidays.analog.analog_holidays import (
    build_analog_ranking_table,
    build_run_summary,
    plot_analog_pair_sequences,
    plot_batch_inference_grid,
    plot_batch_pair_sequences_grid,
    plot_forecast_diagnostics,
    plot_ranked_analog_profiles,
    prepare_audit_working_copy,
    run_analog_holidays,
    run_analog_holidays_batch,
    tune_analog_holidays_optuna,
)

pd.set_option('display.max_rows', 50)
pd.set_option('display.max_columns', 20)

## Parameters

Adjust the series, target date, and analog hyperparameters for the holiday-only CSV source.

- SOURCE_PATH: path to the holiday CSV file consumed by the notebook.
- UNIQUE_ID: target series used to build analogs and generate the forecast.
- TARGET_DATE: holiday date whose operational forecast window should be estimated.
- FORECAST_START_OFFSET_HOURS: how many hours before the holiday midnight the forecast window starts.
- SEASON_LENGTH: hourly profile length to model, in hours. For this workflow it should match `FORECAST_START_OFFSET_HOURS + 24`.
- SPECIAL_LABELS: labels that define which days are treated as special candidates when selecting analogs.
- K: number of special neighbors kept after ranking X against Y by similarity; None uses every filtered candidate.
- OPTUNA_MIN_K: lower bound of the `k` search range explored by Optuna during rolling tuning.
- TYPEDIST: metric used to rank holiday candidates against Y; supports pearson, euclidian, and dtw.
- TYPEREG: regressor type used in the analog reconstruction step; supports PCR, PLS, RidgeReg, LassoReg, RF, OLSstep, and LGBM when `lightgbm` is installed.
- SCALE_METHOD: optional preprocessing transform applied before neighbor selection and regression, then inverted back to the original demand scale for the final forecast. Supported values: `None`, `standard`, `minmax`.
- N_COMPONENTS: number of components for dimensionality-reduction methods such as PCR or PLS; ignored by tree-based regressors.
- REGRESSOR_PARAMS: optional dict of model-specific constructor hyperparameters, mainly for RF or LGBM when running a manual configuration outside Optuna.
- LEVELS: prediction interval levels to compute for the forecast.
- MIN_SPECIAL_POINTS: minimum number of hours flagged as special inside a candidate block.
- MIN_EVENT_GAP: minimum separation between consecutive special events to avoid overly overlapping candidates.
- MAX_EVENTS: maximum number of special events used as the analog bank; None uses all available events.
- SELECTOR_FEATURES_PATH / CLUSTER_COLUMN / USE_CLUSTER / MATCH_TARGET_CLUSTER: optional selector-cluster filter. When `USE_CLUSTER=False`, the analog algorithm chooses neighbors from the full historical holiday pool without restricting to `F/G/H`.
- MAX_PLOTTED_ANALOGS: maximum number of analogs shown in the comparative plots.

In [ ]:
from datetime import datetime

# Original CSV ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â never modified by this notebook.
_ORIGINAL_SOURCE = PROJECT_ROOT / 'analog_holidays' / 'holidays' / 'holiday_demand_mx.csv'
_WORKING_SOURCE_DIR = _ORIGINAL_SOURCE.parent / 'working'

def _ensure_working_source_path(source_path=SOURCE_PATH if 'SOURCE_PATH' in globals() else None):
    candidate = None if source_path is None else Path(source_path)
    if candidate is not None and candidate.exists():
        return candidate
    refreshed = prepare_audit_working_copy(
        _ORIGINAL_SOURCE,
        working_dir=_WORKING_SOURCE_DIR,
        prefix='holiday_demand_mx',
        reuse_today=True,
    )
    print(f'Working copy refreshed: {refreshed.name}')
    return refreshed

# Create or reuse a working copy for this notebook session.
SOURCE_PATH = _ensure_working_source_path()
print(f'Working copy: {Path(SOURCE_PATH).name}')

UNIQUE_IDS = analog_holidays_module.get_available_unique_ids(SOURCE_PATH)
if not UNIQUE_IDS:
    raise ValueError(f'No unique_id values were found in {Path(SOURCE_PATH).name}.')

UNIQUE_ID = 'SEN_demand_SIN'
if UNIQUE_ID not in UNIQUE_IDS:
    UNIQUE_ID = UNIQUE_IDS[0]

print(f'Series to forecast: {len(UNIQUE_IDS)}')
print(', '.join(UNIQUE_IDS))
print(f'Detail view series: {UNIQUE_ID}')

In [ ]:
SPECIAL_LABELS = ('holiday',)
FORECAST_START_OFFSET_HOURS = 14
SEASON_LENGTH = 38  # 14 h pre-holiday + 24 h holiday
K = 100
OPTUNA_MIN_K = 1
TYPEDIST = 'pearson'
TYPEREG = 'PCR'
SCALE_METHOD = None
OPTUNA_SCALE_METHOD_CHOICES = [None, 'standard', 'minmax']
N_COMPONENTS = 3 #
REGRESSOR_PARAMS = {}
LEVELS = [80, 95]
MIN_SPECIAL_POINTS = 24  # require the 24 holiday hours inside each 38-h candidate window
MIN_EVENT_GAP = 24
MAX_EVENTS = None
MAX_PLOTTED_ANALOGS = 10
RECENT_WEEKEND_ANALOGS = 0

SELECTOR_FEATURES_PATH = PROJECT_ROOT / 'analog_holidays' / 'holidays' / 'holiday_selector_features.csv'
CLUSTER_COLUMN = 'analog_cluster'
USE_CLUSTER = True
MATCH_TARGET_CLUSTER = bool(USE_CLUSTER)

OPTUNA_N_TRIALS = 25
OPTUNA_TIMEOUT_SEC = 900
OPTUNA_MAX_EVAL_DATES = 12
OPTUNA_RANDOM_SEED = 42

HOURLY_FACTOR_ANALOGS = 4


In [ ]:
if OPTUNA_SCALE_METHOD_CHOICES is not None:
    valid_scale_methods = {None, 'standard', 'minmax'}
    invalid_scale_methods = [
        value for value in OPTUNA_SCALE_METHOD_CHOICES
        if value not in valid_scale_methods
    ]
    if invalid_scale_methods:
        raise ValueError(
            f'Unsupported OPTUNA_SCALE_METHOD_CHOICES: {invalid_scale_methods}'
        )

if int(OPTUNA_MIN_K) < 1:
    raise ValueError(f'OPTUNA_MIN_K must be >= 1, got {OPTUNA_MIN_K}.')

print(f'USE_CLUSTER: {USE_CLUSTER}')
print(f'MATCH_TARGET_CLUSTER: {MATCH_TARGET_CLUSTER}')
print(f'OPTUNA_MIN_K: {OPTUNA_MIN_K}')
print(f'SCALE_METHOD fixed fallback: {SCALE_METHOD}')
print(f'OPTUNA scale choices: {OPTUNA_SCALE_METHOD_CHOICES}')

In [ ]:
TARGET_DATES_2025 = [
    ('2025-01-01', "New Year's Day"),
    ('2025-02-03', 'Constitution Day'),
    ('2025-03-17', "Benito Juarez's Birthday"),
    ('2025-04-17', 'Maundy Thursday'),
    ('2025-04-18', 'Good Friday'),
    ('2025-04-19', 'Holy Saturday'),
    ('2025-05-01', 'Labor Day'),
    ('2025-09-16', 'Independence Day'),
    ('2025-11-17', 'Mexican Revolution Day'),
    ('2025-12-24', 'Christmas Eve'),
    ('2025-12-25', 'Christmas Day'),
    ('2025-12-31', "New Year's Eve"),
    # ===== 2026 =====
    ('2026-01-01', "New Year's Day"),    
    ('2026-02-02', 'Constitution Day'),
    ('2026-03-16', "Benito Juarez's Birthday"),
    ('2026-04-02', 'Maundy Thursday'),
    ('2026-04-03', 'Good Friday'),
    ('2026-04-04', 'Holy Saturday'),
    ('2026-05-01', 'Labor Day'),
    # ('2026-09-16', 'Independence Day'),
    # ('2026-11-16', 'Mexican Revolution Day'),
    # ('2026-12-24', 'Christmas Eve'),
    # ('2026-12-25', 'Christmas Day'),
    # ('2026-12-31', "New Year's Eve"),
]

TARGET_DATE = TARGET_DATES_2025[-1][0]

from matplotlib.backends.backend_pdf import PdfPages

RESULTS_DIR = PROJECT_ROOT / 'analog_holidays' / 'Holiday_results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MULTIPAGE_PDF_PATH = RESULTS_DIR / 'all_holiday_graphs.pdf'
PDF_FIGURES = {}

def _pdf_safe_name(value):
    safe = ''.join(ch if ch.isalnum() or ch in ('-', '_') else '_' for ch in str(value))
    while '__' in safe:
        safe = safe.replace('__', '_')
    return safe.strip('_') or 'figure'

def export_figure_pdf(fig_obj, stem):
    safe_stem = _pdf_safe_name(stem)
    single_path = RESULTS_DIR / f'{safe_stem}.pdf'
    fig_obj.savefig(single_path, format='pdf', bbox_inches='tight')
    PDF_FIGURES[safe_stem] = fig_obj
    with PdfPages(MULTIPAGE_PDF_PATH) as pdf:
        for saved_fig in PDF_FIGURES.values():
            pdf.savefig(saved_fig, bbox_inches='tight')
    print(f'Saved PDF: {single_path}')
    print(f'Updated multipage PDF: {MULTIPAGE_PDF_PATH}')
    return single_path, MULTIPAGE_PDF_PATH

print(f'PDF output folder: {RESULTS_DIR}')

In [ ]:
selector_features_df = pd.read_csv(SELECTOR_FEATURES_PATH, parse_dates=['date'])
selector_features_df['date'] = pd.to_datetime(selector_features_df['date']).dt.normalize()
if 'unique_id' not in selector_features_df.columns:
    raise ValueError(
        f'{SELECTOR_FEATURES_PATH.name} must contain a unique_id column. '
        'Re-export the selector from M_identify_holidays.ipynb.'
    )

selector_features_current_df = selector_features_df.loc[
    selector_features_df['unique_id'].astype(str) == str(UNIQUE_ID)
].copy()
if selector_features_current_df.empty:
    raise ValueError(
        f'No selector rows were found for UNIQUE_ID={UNIQUE_ID!r} in {SELECTOR_FEATURES_PATH.name}.'
    )

target_ts = pd.Timestamp(TARGET_DATE).normalize()

target_cluster_df = selector_features_current_df.loc[
    selector_features_current_df['date'] == target_ts,
    ['unique_id', 'date', 'holiday_name', 'holiday_day_type', CLUSTER_COLUMN],
]

if MATCH_TARGET_CLUSTER:
    if target_cluster_df.empty:
        raise ValueError(
            f'No selector cluster was found for UNIQUE_ID={UNIQUE_ID!r} and TARGET_DATE={target_ts.date()} '
            f'in {SELECTOR_FEATURES_PATH.name}.'
        )
    TARGET_ANALOG_CLUSTER = target_cluster_df.iloc[0][CLUSTER_COLUMN]
    eligible_cluster_analogs_df = selector_features_current_df.loc[
        (selector_features_current_df[CLUSTER_COLUMN] == TARGET_ANALOG_CLUSTER)
        & (selector_features_current_df['date'] < target_ts),
        ['unique_id', 'date', 'holiday_name', 'anchor_holiday_name', 'holiday_day_type', CLUSTER_COLUMN],
    ].sort_values('date').reset_index(drop=True)
    print(
        f'UNIQUE_ID={UNIQUE_ID} | TARGET_DATE={target_ts.date()} -> analog_cluster={TARGET_ANALOG_CLUSTER} '
        f'| eligible historical analog dates={len(eligible_cluster_analogs_df)}'
    )
    display(target_cluster_df)
else:
    TARGET_ANALOG_CLUSTER = pd.NA
    eligible_cluster_analogs_df = selector_features_current_df.loc[
        selector_features_current_df['date'] < target_ts,
        ['unique_id', 'date', 'holiday_name', 'anchor_holiday_name', 'holiday_day_type', CLUSTER_COLUMN],
    ].sort_values('date').reset_index(drop=True)
    print(
        f'UNIQUE_ID={UNIQUE_ID} | TARGET_DATE={target_ts.date()} -> cluster filter disabled '
        f'| historical holiday rows={len(eligible_cluster_analogs_df)}'
    )
    if not target_cluster_df.empty:
        display(target_cluster_df)

display(eligible_cluster_analogs_df)

In [ ]:
SOURCE_PATH = _ensure_working_source_path(SOURCE_PATH)

rolling_target_items = [
    (pd.Timestamp(target_date).date().isoformat(), holiday_label)
    for target_date, holiday_label in TARGET_DATES_2025
]

series_unique_ids = list(UNIQUE_IDS) if 'UNIQUE_IDS' in globals() else [UNIQUE_ID]
if not series_unique_ids:
    raise ValueError('UNIQUE_IDS is empty.')
if UNIQUE_ID not in series_unique_ids:
    UNIQUE_ID = series_unique_ids[0]
series_unique_ids = [str(value) for value in series_unique_ids]
selector_lookup_df = selector_features_df.copy()
selector_lookup_df['unique_id'] = selector_lookup_df['unique_id'].astype(str)

selector_cluster_lookup_by_id = {
    series_unique_id: (
        selector_lookup_df.loc[selector_lookup_df['unique_id'] == series_unique_id]
        .dropna(subset=[CLUSTER_COLUMN])
        .drop_duplicates(subset=['date'], keep='last')
        .set_index('date')[CLUSTER_COLUMN]
        .to_dict()
    )
    for series_unique_id in series_unique_ids
}

selector_anchor_lookup_by_id = {
    series_unique_id: (
        selector_lookup_df.loc[selector_lookup_df['unique_id'] == series_unique_id]
        .dropna(subset=['anchor_holiday_name'])
        .drop_duplicates(subset=['date'], keep='last')
        .set_index('date')['anchor_holiday_name']
        .to_dict()
    )
    for series_unique_id in series_unique_ids
}

def _summary_param(summary_df, param_name, default=np.nan):
    matches = summary_df.loc[summary_df['param'] == param_name, 'value']
    return matches.iloc[0] if not matches.empty else default

rolling_optuna_results = {}
rolling_runs = {}
rolling_rows = []

for unique_id in series_unique_ids:
    print(f'=== UNIQUE_ID={unique_id} ===')
    series_cluster_lookup = selector_cluster_lookup_by_id.get(str(unique_id), {})
    series_anchor_lookup = selector_anchor_lookup_by_id.get(str(unique_id), {})
    series_optuna_results = {}
    series_runs = {}

    for target_date, holiday_label in rolling_target_items:
        target_ts = pd.Timestamp(target_date).normalize()
        target_cluster = series_cluster_lookup.get(target_ts, pd.NA)
        anchor_holiday_name = series_anchor_lookup.get(target_ts, holiday_label)
        if pd.isna(anchor_holiday_name):
            anchor_holiday_name = holiday_label
        anchor_holiday_name = str(anchor_holiday_name)
        print(f'[{unique_id}] [{target_date}] tuning and forecasting [{anchor_holiday_name}]...')

        try:
            tuning_result = tune_analog_holidays_optuna(
                unique_id=unique_id,
                source_path=SOURCE_PATH,
                train_end=target_ts,
                season_length=SEASON_LENGTH,
                forecast_start_offset_hours=FORECAST_START_OFFSET_HOURS,
                initial_k=K,
                initial_typedist=TYPEDIST,
                initial_typereg=TYPEREG,
                scale_method=SCALE_METHOD,
                scale_method_choices=OPTUNA_SCALE_METHOD_CHOICES,
                initial_n_components=N_COMPONENTS,
                initial_regressor_params=REGRESSOR_PARAMS,
                optuna_min_k=OPTUNA_MIN_K,
                n_trials=OPTUNA_N_TRIALS,
                timeout_sec=OPTUNA_TIMEOUT_SEC,
                max_eval_dates=OPTUNA_MAX_EVAL_DATES,
                random_seed=OPTUNA_RANDOM_SEED,
                special_labels=SPECIAL_LABELS,
                min_special_points=MIN_SPECIAL_POINTS,
                min_event_gap=MIN_EVENT_GAP,
                max_events=MAX_EVENTS,
                selector_features_path=SELECTOR_FEATURES_PATH,
                cluster_column=CLUSTER_COLUMN,
                match_target_cluster=MATCH_TARGET_CLUSTER,
                recent_weekend_analogs=RECENT_WEEKEND_ANALOGS,
            )
            series_optuna_results[target_date] = tuning_result

            best_config = tuning_result.best_config
            best_k_range = tuple(best_config.get('k_range', (np.nan, np.nan)))
            best_scale_method = best_config.get('scale_method', SCALE_METHOD)
            best_regressor_params = dict(best_config.get('regressor_params', {}))
            run = run_analog_holidays(
                unique_id=unique_id,
                target_date=target_ts,
                source_path=SOURCE_PATH,
                season_length=SEASON_LENGTH,
                forecast_start_offset_hours=FORECAST_START_OFFSET_HOURS,
                k=int(best_config['k']),
                typedist=str(best_config['typedist']),
                typereg=str(best_config['typereg']),
                scale_method=best_scale_method,
                n_components=int(best_config['n_components']),
                regressor_params=best_regressor_params,
                levels=LEVELS,
                special_labels=SPECIAL_LABELS,
                min_special_points=MIN_SPECIAL_POINTS,
                min_event_gap=MIN_EVENT_GAP,
                max_events=MAX_EVENTS,
                expected_target_label=None,
                selector_features_path=SELECTOR_FEATURES_PATH,
                cluster_column=CLUSTER_COLUMN,
                match_target_cluster=MATCH_TARGET_CLUSTER,
                recent_weekend_analogs=RECENT_WEEKEND_ANALOGS,
            )
            series_runs[target_date] = run

            mae_window = np.nan
            mape_window_pct = np.nan
            if run.actual_profile is not None:
                mae_window = float(np.mean(np.abs(run.forecast_profile - run.actual_profile)))
                denom = np.where(np.abs(run.actual_profile) > 1e-9, np.abs(run.actual_profile), np.nan)
                ape_pct = np.abs(run.forecast_profile - run.actual_profile) / denom * 100.0
                if np.isfinite(ape_pct).any():
                    mape_window_pct = float(np.nanmean(ape_pct))

            rolling_rows.append({
                'unique_id': unique_id,
                'target_date': target_date,
                'holiday_label': holiday_label,
                'analog_cluster': target_cluster,
                'filter_by_cluster': bool(MATCH_TARGET_CLUSTER),
                'train_end': target_date,
                'eligible_tuning_dates': len(tuning_result.eligible_dates),
                'target_exists': run.target_exists,
                'target_has_complete_profile': run.target_has_complete_profile,
                'selected_analogs': len(run.positions),
                'fail': run.fail,
                'optuna_k_min': best_k_range[0],
                'optuna_k_max': best_k_range[1],
                'k': run.k,
                'typedist': run.typedist,
                'typereg': run.typereg,
                'scale_method': run.scale_method,
                'n_components': run.n_components,
                'regressor_params': best_regressor_params,
                'forecast_start': run.forecast_start,
                'forecast_end': run.forecast_end,
                'mae_window': mae_window,
                'mape_window_pct': mape_window_pct,
                'tuning_best_mean_mae': _summary_param(tuning_result.summary_df, 'best_mean_mae'),
                'tuning_best_mean_mape_pct': _summary_param(tuning_result.summary_df, 'best_mean_mape_pct'),
                'error': None,
            })
            print(
                f'[{unique_id}] [{target_date}] cluster={target_cluster} | '
                f'k-range(optuna-param)=[{best_k_range[0]}, {best_k_range[1]}] | '
                f'k(optuna)={run.k} | '
                f'typereg(optuna)={run.typereg} | '
                f'typedist(optuna)={run.typedist} | '
                f'scale_method(optuna)={run.scale_method} | '
                f'MAPE(final)={mape_window_pct:.2f}%'
            )
        except Exception as exc:
            rolling_rows.append({
                'unique_id': unique_id,
                'target_date': target_date,
                'holiday_label': holiday_label,
                'analog_cluster': target_cluster,
                'filter_by_cluster': bool(MATCH_TARGET_CLUSTER),
                'train_end': target_date,
                'eligible_tuning_dates': np.nan,
                'target_exists': False,
                'target_has_complete_profile': False,
                'selected_analogs': 0,
                'fail': True,
                'optuna_k_min': np.nan,
                'optuna_k_max': np.nan,
                'k': np.nan,
                'typedist': pd.NA,
                'typereg': pd.NA,
                'scale_method': pd.NA,
                'n_components': np.nan,
                'regressor_params': {},
                'forecast_start': pd.NaT,
                'forecast_end': pd.NaT,
                'mae_window': np.nan,
                'mape_window_pct': np.nan,
                'tuning_best_mean_mae': np.nan,
                'tuning_best_mean_mape_pct': np.nan,
                'error': str(exc),
            })
            print(f'[{unique_id}] [{target_date}] ERROR: {exc}')

    rolling_optuna_results[unique_id] = series_optuna_results
    rolling_runs[unique_id] = series_runs

rolling_daily_table = pd.DataFrame(rolling_rows)

batch_result_2025_all = {}
if not rolling_daily_table.empty:
    for series_unique_id, series_run_map in rolling_runs.items():
        series_results_df = (
            rolling_daily_table
            .loc[rolling_daily_table['unique_id'] == series_unique_id]
            .reset_index(drop=True)
        )
        if series_results_df.empty:
            continue
        metric_summary_columns = [
            column for column in [
                'mae_holiday24_bias_adjusted',
                'mape_holiday24_bias_adjusted_pct',
                'mae_holiday24_raw',
                'mape_holiday24_raw_pct',
                'mae_window_bias_adjusted',
                'mape_window_bias_adjusted_pct',
                'mae_window',
                'mape_window_pct',
            ]
            if column in series_results_df.columns
        ]
        batch_result_2025_all[series_unique_id] = analog_holidays_module.AnalogHolidayBatchResult(
            target_items=rolling_target_items,
            runs=series_run_map,
            results_df=series_results_df,
            metric_summary_df=series_results_df[metric_summary_columns].describe(include='all'),
        )

batch_result_2025 = batch_result_2025_all.get(UNIQUE_ID)
rolling_daily_table

In [ ]:
# Ajuste dinámico 38h: estima la forma intradiaria sobre la ventana completa usando hasta los 4 análogos más similares disponibles.
HOURLY_FACTOR_ANALOGS = 4
BIAS_HEAD_HOURS = int(FORECAST_START_OFFSET_HOURS)
BIAS_TAIL_HOURS = int(SEASON_LENGTH - BIAS_HEAD_HOURS)

if BIAS_HEAD_HOURS <= 0 or BIAS_TAIL_HOURS <= 0:
    raise ValueError(
        f'Hourly holiday adjustment requires a valid 14h/24h split inside the 38h window. '
        f'Got FORECAST_START_OFFSET_HOURS={FORECAST_START_OFFSET_HOURS} and SEASON_LENGTH={SEASON_LENGTH}.'
    )
if 'rolling_daily_table' not in globals() or rolling_daily_table.empty:
    raise ValueError('Run the rolling study cell first to build rolling_daily_table.')
if 'rolling_runs' not in globals() or not rolling_runs:
    raise ValueError('Run the rolling study cell first to build rolling_runs.')

def _mean_abs_error(actual, forecast):
    return float(np.mean(np.abs(actual - forecast)))

def _mean_ape_pct(actual, forecast):
    denom = np.where(np.abs(actual) > 1e-9, np.abs(actual), np.nan)
    ape_pct = np.abs(actual - forecast) / denom * 100.0
    return float(np.nanmean(ape_pct)) if np.isfinite(ape_pct).any() else np.nan

def _mean_pct_error(actual, forecast):
    denom = np.where(np.abs(actual) > 1e-9, np.abs(actual), np.nan)
    pe_pct = (actual - forecast) / denom * 100.0
    return float(np.nanmean(pe_pct)) if np.isfinite(pe_pct).any() else np.nan

def _mean_bias(actual, forecast):
    return float(np.mean(actual - forecast))

def _build_window_metrics(actual, forecast, label):
    return {
        f'mae_{label}': _mean_abs_error(actual, forecast),
        f'mape_{label}_pct': _mean_ape_pct(actual, forecast),
        f'mpe_{label}_pct': _mean_pct_error(actual, forecast),
        f'bias_{label}': _mean_bias(actual, forecast),
    }

def _safe_corr(series_a, series_b):
    clean_df = pd.DataFrame({'a': series_a, 'b': series_b}).dropna()
    if len(clean_df) < 2 or clean_df['a'].nunique() < 2 or clean_df['b'].nunique() < 2:
        return np.nan
    return float(clean_df['a'].corr(clean_df['b']))

def _apply_hourly_factor_model(forecast_profile, factor_model, expected_hours):
    forecast_profile = np.asarray(forecast_profile, dtype=np.float64)
    resolved_expected_hours = int(expected_hours)
    if forecast_profile.shape[0] != resolved_expected_hours:
        raise ValueError(
            f'Expected a {resolved_expected_hours}-hour forecast, got shape {forecast_profile.shape}.'
        )
    model_window_hours = int(factor_model.get('window_hours', resolved_expected_hours))
    if model_window_hours != resolved_expected_hours:
        raise ValueError(
            f'Bias factor model window_hours={model_window_hours} does not match expected_hours={resolved_expected_hours}.'
        )
    forecast_window_mean = float(np.mean(forecast_profile))
    if factor_model['train_samples'] <= 0:
        return forecast_profile.copy(), forecast_window_mean
    adjusted_profile = forecast_window_mean * (1.0 + factor_model['hourly_factors'])
    return adjusted_profile.astype(np.float64), forecast_window_mean

bias_adjusted_profiles = {}
bias_rows = []

for unique_id, series_runs in rolling_runs.items():
    ordered_target_dates = sorted(series_runs, key=lambda value: pd.Timestamp(value))

    for target_date in ordered_target_dates:
        run = series_runs[target_date]
        if run.actual_profile is None or len(run.forecast_profile) != SEASON_LENGTH:
            continue

        full_actual = run.actual_profile.copy()
        full_forecast = run.forecast_profile.copy()
        head_actual = full_actual[:BIAS_HEAD_HOURS]
        head_forecast = full_forecast[:BIAS_HEAD_HOURS]
        tail_actual = full_actual[BIAS_HEAD_HOURS:]
        tail_forecast = full_forecast[BIAS_HEAD_HOURS:]

        factor_model = analog_holidays_module.fit_hourly_bias_factor_model(
            neighbor_profiles=run.neighbors2,
            window_hours=SEASON_LENGTH,
            max_analogs=HOURLY_FACTOR_ANALOGS,
        )
        adjusted_full_forecast, forecast_window_mean_38 = _apply_hourly_factor_model(
            full_forecast,
            factor_model,
            expected_hours=SEASON_LENGTH,
        )
        adjusted_head_forecast = adjusted_full_forecast[:BIAS_HEAD_HOURS].copy()
        adjusted_tail_forecast = adjusted_full_forecast[BIAS_HEAD_HOURS:].copy()
        forecast_daily_mean_24 = float(np.mean(tail_forecast))
        predicted_full_bias_profile = adjusted_full_forecast - full_forecast
        predicted_head_bias_mean = float(np.mean(predicted_full_bias_profile[:BIAS_HEAD_HOURS]))
        predicted_tail_bias_mean = float(np.mean(predicted_full_bias_profile[BIAS_HEAD_HOURS:]))
        predicted_full_bias_mean = float(np.mean(predicted_full_bias_profile))

        metrics_14 = _build_window_metrics(head_actual, head_forecast, '14')
        metrics_24 = _build_window_metrics(tail_actual, tail_forecast, '24')
        metrics_38 = _build_window_metrics(full_actual, full_forecast, '38')
        metrics_14_bias_adjusted = _build_window_metrics(head_actual, adjusted_head_forecast, '14_bias_adjusted')
        metrics_24_bias_adjusted = _build_window_metrics(tail_actual, adjusted_tail_forecast, '24_bias_adjusted')
        metrics_38_bias_adjusted = _build_window_metrics(full_actual, adjusted_full_forecast, '38_bias_adjusted')
        predicted_bias_14 = predicted_head_bias_mean
        predicted_bias_24 = predicted_tail_bias_mean
        predicted_bias_38 = predicted_full_bias_mean
        bias_14_prediction_error = predicted_bias_14 - metrics_14['bias_14']
        bias_24_prediction_error = predicted_bias_24 - metrics_24['bias_24']
        bias_38_prediction_error = predicted_bias_38 - metrics_38['bias_38']

        bias_adjusted_profiles[(unique_id, target_date)] = {
            'full_actual': full_actual.copy(),
            'full_forecast': full_forecast.copy(),
            'adjusted_full_forecast': adjusted_full_forecast.copy(),
            'head_actual': head_actual.copy(),
            'head_forecast': head_forecast.copy(),
            'adjusted_head_forecast': adjusted_head_forecast.copy(),
            'tail_actual': tail_actual.copy(),
            'tail_forecast': tail_forecast.copy(),
            'adjusted_tail_forecast': adjusted_tail_forecast.copy(),
            'hourly_adjustment_factors': factor_model['hourly_factors'].copy(),
            'hourly_factor_analog_count': factor_model['train_samples'],
            'hourly_factor_requested_analogs': factor_model['requested_analogs'],
            'hourly_factor_available_neighbors': factor_model['available_neighbor_profiles'],
            'hourly_factor_selected_analogs': factor_model['selected_analogs'],
            'forecast_window_mean_38': forecast_window_mean_38,
            'forecast_daily_mean_24': forecast_daily_mean_24,
        }

        bias_rows.append({
            'unique_id': unique_id,
            'target_date': target_date,
            'bias_head_hours': BIAS_HEAD_HOURS,
            'bias_tail_hours': BIAS_TAIL_HOURS,
            'hourly_factor_analog_count': factor_model['train_samples'],
            'hourly_factor_requested_analogs': factor_model['requested_analogs'],
            'hourly_factor_available_neighbors': factor_model['available_neighbor_profiles'],
            'hourly_factor_selected_analogs': factor_model['selected_analogs'],
            'hourly_factor_mean_abs': factor_model['factor_mean_abs'],
            'forecast_window_mean_38': forecast_window_mean_38,
            'forecast_daily_mean_24': forecast_daily_mean_24,
            **metrics_38,
            **metrics_14,
            **metrics_24,
            **metrics_14_bias_adjusted,
            **metrics_24_bias_adjusted,
            **metrics_38_bias_adjusted,
            'head_bias_mean': metrics_14['bias_14'],
            'tail_bias_mean': metrics_24['bias_24'],
            'bias_model_method': factor_model['method'],
            'bias_train_samples': factor_model['train_samples'],
            'bias_train_window_mean': factor_model['window_mean_train'],
            'bias_train_head_tail_corr': factor_model['head_tail_corr_train'],
            'bias_model_intercept': factor_model['intercept'],
            'bias_model_slope': factor_model['slope'],
            'predicted_head_bias_mean': predicted_head_bias_mean,
            'predicted_tail_bias_mean': predicted_tail_bias_mean,
            'predicted_full_bias_mean': predicted_full_bias_mean,
            'predicted_bias_14': predicted_bias_14,
            'predicted_bias_24': predicted_bias_24,
            'predicted_bias_38': predicted_bias_38,
            'bias_14_prediction_error': bias_14_prediction_error,
            'bias_24_prediction_error': bias_24_prediction_error,
            'bias_38_prediction_error': bias_38_prediction_error,
            'mae_head14': metrics_14['mae_14'],
            'mape_head14_pct': metrics_14['mape_14_pct'],
            'mae_head14_bias_adjusted': metrics_14_bias_adjusted['mae_14_bias_adjusted'],
            'mape_head14_bias_adjusted_pct': metrics_14_bias_adjusted['mape_14_bias_adjusted_pct'],
            'mae_holiday24_raw': metrics_24['mae_24'],
            'mape_holiday24_raw_pct': metrics_24['mape_24_pct'],
            'mae_holiday24_bias_adjusted': metrics_24_bias_adjusted['mae_24_bias_adjusted'],
            'mape_holiday24_bias_adjusted_pct': metrics_24_bias_adjusted['mape_24_bias_adjusted_pct'],
            'mae_window_bias_adjusted': metrics_38_bias_adjusted['mae_38_bias_adjusted'],
            'mape_window_bias_adjusted_pct': metrics_38_bias_adjusted['mape_38_bias_adjusted_pct'],
        })

rolling_bias_adjustment_df = pd.DataFrame(bias_rows)
if rolling_bias_adjustment_df.empty:
    raise ValueError('No complete rolling runs were available to build the hourly-adjustment study.')

rolling_daily_table = rolling_daily_table.merge(
    rolling_bias_adjustment_df,
    on=['unique_id', 'target_date'],
    how='left',
    validate='one_to_one',
)

rolling_daily_table['mae_holiday24_improvement'] = (
    rolling_daily_table['mae_holiday24_raw'] - rolling_daily_table['mae_holiday24_bias_adjusted']
)
rolling_daily_table['mape_holiday24_improvement_pct'] = (
    rolling_daily_table['mape_holiday24_raw_pct'] - rolling_daily_table['mape_holiday24_bias_adjusted_pct']
)
rolling_daily_table['mape_head14_improvement_pct'] = (
    rolling_daily_table['mape_head14_pct'] - rolling_daily_table['mape_head14_bias_adjusted_pct']
)
rolling_daily_table['mape_window_improvement_pct'] = (
    rolling_daily_table['mape_window_pct'] - rolling_daily_table['mape_window_bias_adjusted_pct']
)
rolling_daily_table['bias_14_prediction_abs_error'] = rolling_daily_table['bias_14_prediction_error'].abs()
rolling_daily_table['bias_24_prediction_abs_error'] = rolling_daily_table['bias_24_prediction_error'].abs()
rolling_daily_table['bias_38_prediction_abs_error'] = rolling_daily_table['bias_38_prediction_error'].abs()

if 'batch_result_2025_all' in globals() and batch_result_2025_all:
    for series_unique_id, series_batch_result in batch_result_2025_all.items():
        series_batch_result.results_df = (
            rolling_daily_table
            .loc[rolling_daily_table['unique_id'] == series_unique_id]
            .reset_index(drop=True)
        )
        metric_summary_columns = [
            column for column in [
                'mae_38',
                'mape_38_pct',
                'mpe_38_pct',
                'bias_38',
                'mae_14',
                'mape_14_pct',
                'mpe_14_pct',
                'bias_14',
                'mae_24',
                'mape_24_pct',
                'mpe_24_pct',
                'bias_24',
                'predicted_bias_38',
                'bias_38_prediction_error',
                'predicted_bias_24',
                'bias_24_prediction_error',
                'hourly_factor_analog_count',
                'hourly_factor_requested_analogs',
                'hourly_factor_available_neighbors',
                'hourly_factor_selected_analogs',
                'hourly_factor_mean_abs',
                'forecast_window_mean_38',
                'forecast_daily_mean_24',
                'mae_holiday24_bias_adjusted',
                'mape_holiday24_bias_adjusted_pct',
                'mpe_24_bias_adjusted_pct',
                'bias_24_bias_adjusted',
                'mae_holiday24_raw',
                'mape_holiday24_raw_pct',
                'mae_window_bias_adjusted',
                'mape_window_bias_adjusted_pct',
                'mpe_38_bias_adjusted_pct',
                'bias_38_bias_adjusted',
            ]
            if column in series_batch_result.results_df.columns
        ]
        series_batch_result.metric_summary_df = series_batch_result.results_df[metric_summary_columns].describe(include='all')
bias_summary_df = (
    rolling_daily_table
    .groupby('unique_id', dropna=False)
    .agg(
        rows=('target_date', 'size'),
        rows_with_train_history=('bias_train_samples', lambda values: int((values > 0).sum())),
        median_bias_train_samples=('bias_train_samples', 'median'),
        mean_mae_38=('mae_38', 'mean'),
        mean_mae_14=('mae_14', 'mean'),
        mean_mae_24=('mae_24', 'mean'),
        mean_mape_38_pct=('mape_38_pct', 'mean'),
        mean_mape_14_pct=('mape_14_pct', 'mean'),
        mean_mape_24_pct=('mape_24_pct', 'mean'),
        mean_mpe_38_pct=('mpe_38_pct', 'mean'),
        mean_mpe_14_pct=('mpe_14_pct', 'mean'),
        mean_mpe_24_pct=('mpe_24_pct', 'mean'),
        mean_bias_38=('bias_38', 'mean'),
        mean_bias_14=('bias_14', 'mean'),
        mean_bias_24=('bias_24', 'mean'),
        mean_hourly_factor_analog_count=('hourly_factor_analog_count', 'mean'),
        mean_hourly_factor_requested_analogs=('hourly_factor_requested_analogs', 'mean'),
        mean_hourly_factor_available_neighbors=('hourly_factor_available_neighbors', 'mean'),
        mean_hourly_factor_selected_analogs=('hourly_factor_selected_analogs', 'mean'),
        mean_hourly_factor_mean_abs=('hourly_factor_mean_abs', 'mean'),
        mean_predicted_bias_38=('predicted_bias_38', 'mean'),
        mean_bias_38_prediction_error=('bias_38_prediction_error', 'mean'),
        mean_predicted_bias_24=('predicted_bias_24', 'mean'),
        mean_bias_24_prediction_error=('bias_24_prediction_error', 'mean'),
    )
    .reset_index()
)

eligible_bias_rows = rolling_daily_table['bias_train_samples'] > 0
bias_linkage_summary_rows = []
for series_unique_id, series_df in rolling_daily_table.groupby('unique_id', dropna=False):
    eligible_series_df = series_df.loc[series_df['bias_train_samples'] > 0].copy()
    bias_linkage_summary_rows.append({
        'unique_id': series_unique_id,
        'rows_with_train_history': int(len(eligible_series_df)),
        'corr_bias_14_vs_24': _safe_corr(eligible_series_df['bias_14'], eligible_series_df['bias_24']),
        'corr_mpe_14_vs_24_pct': _safe_corr(eligible_series_df['mpe_14_pct'], eligible_series_df['mpe_24_pct']),
        'mean_hourly_factor_analog_count': float(eligible_series_df['hourly_factor_analog_count'].mean()) if not eligible_series_df.empty else np.nan,
        'mean_hourly_factor_requested_analogs': float(eligible_series_df['hourly_factor_requested_analogs'].mean()) if not eligible_series_df.empty else np.nan,
        'mean_hourly_factor_available_neighbors': float(eligible_series_df['hourly_factor_available_neighbors'].mean()) if not eligible_series_df.empty else np.nan,
        'mean_hourly_factor_selected_analogs': float(eligible_series_df['hourly_factor_selected_analogs'].mean()) if not eligible_series_df.empty else np.nan,
        'mean_hourly_factor_mean_abs': float(eligible_series_df['hourly_factor_mean_abs'].mean()) if not eligible_series_df.empty else np.nan,
        'mean_predicted_bias_38': float(eligible_series_df['predicted_bias_38'].mean()) if not eligible_series_df.empty else np.nan,
        'mean_actual_bias_38': float(eligible_series_df['bias_38'].mean()) if not eligible_series_df.empty else np.nan,
        'mae_bias_prediction_38': float(np.mean(np.abs(eligible_series_df['bias_38_prediction_error']))) if not eligible_series_df.empty else np.nan,
        'mean_bias_prediction_error_38': float(eligible_series_df['bias_38_prediction_error'].mean()) if not eligible_series_df.empty else np.nan,
    })
bias_linkage_summary_df = pd.DataFrame(bias_linkage_summary_rows)

overall_bias_delta_df = pd.DataFrame([{
    'rows_total': int(len(rolling_daily_table)),
    'rows_with_train_history': int(eligible_bias_rows.sum()),
    'mean_mape_38_pct': float(rolling_daily_table.loc[eligible_bias_rows, 'mape_38_pct'].mean()),
    'mean_mape_14_pct': float(rolling_daily_table.loc[eligible_bias_rows, 'mape_14_pct'].mean()),
    'mean_mape_24_pct': float(rolling_daily_table.loc[eligible_bias_rows, 'mape_24_pct'].mean()),
    'mean_mae_38': float(rolling_daily_table.loc[eligible_bias_rows, 'mae_38'].mean()),
    'mean_mae_14': float(rolling_daily_table.loc[eligible_bias_rows, 'mae_14'].mean()),
    'mean_mae_24': float(rolling_daily_table.loc[eligible_bias_rows, 'mae_24'].mean()),
    'mean_mpe_38_pct': float(rolling_daily_table.loc[eligible_bias_rows, 'mpe_38_pct'].mean()),
    'mean_mpe_14_pct': float(rolling_daily_table.loc[eligible_bias_rows, 'mpe_14_pct'].mean()),
    'mean_mpe_24_pct': float(rolling_daily_table.loc[eligible_bias_rows, 'mpe_24_pct'].mean()),
    'mean_bias_38': float(rolling_daily_table.loc[eligible_bias_rows, 'bias_38'].mean()),
    'mean_bias_14': float(rolling_daily_table.loc[eligible_bias_rows, 'bias_14'].mean()),
    'mean_bias_24': float(rolling_daily_table.loc[eligible_bias_rows, 'bias_24'].mean()),
    'mean_hourly_factor_analog_count': float(rolling_daily_table.loc[eligible_bias_rows, 'hourly_factor_analog_count'].mean()),
    'mean_hourly_factor_requested_analogs': float(rolling_daily_table.loc[eligible_bias_rows, 'hourly_factor_requested_analogs'].mean()),
    'mean_hourly_factor_available_neighbors': float(rolling_daily_table.loc[eligible_bias_rows, 'hourly_factor_available_neighbors'].mean()),
    'mean_hourly_factor_selected_analogs': float(rolling_daily_table.loc[eligible_bias_rows, 'hourly_factor_selected_analogs'].mean()),
    'mean_hourly_factor_mean_abs': float(rolling_daily_table.loc[eligible_bias_rows, 'hourly_factor_mean_abs'].mean()),
    'mean_predicted_bias_38': float(rolling_daily_table.loc[eligible_bias_rows, 'predicted_bias_38'].mean()),
    'corr_bias_14_vs_24': _safe_corr(rolling_daily_table.loc[eligible_bias_rows, 'bias_14'], rolling_daily_table.loc[eligible_bias_rows, 'bias_24']),
    'holiday24_improved_rows': int((rolling_daily_table['mape_holiday24_improvement_pct'] > 0).sum()),
    'window38_improved_rows': int((rolling_daily_table['mape_window_improvement_pct'] > 0).sum()),
    'window38_worsened_rows': int((rolling_daily_table['mape_window_improvement_pct'] < 0).sum()),
}])

display(overall_bias_delta_df)
display(bias_summary_df)
display(bias_linkage_summary_df)
rolling_daily_table[[
    'unique_id',
    'target_date',
    'analog_cluster',
    'bias_train_samples',
    'bias_model_method',
    'hourly_factor_analog_count',
    'hourly_factor_requested_analogs',
    'hourly_factor_available_neighbors',
    'hourly_factor_selected_analogs',
    'hourly_factor_mean_abs',
    'forecast_window_mean_38',
    'forecast_daily_mean_24',
    'mae_38',
    'mape_38_pct',
    'mpe_38_pct',
    'bias_38',
    'mae_14',
    'mape_14_pct',
    'mpe_14_pct',
    'bias_14',
    'mae_24',
    'mape_24_pct',
    'mpe_24_pct',
    'bias_24',
    'predicted_bias_38',
    'bias_38_prediction_error',
    'predicted_bias_24',
    'bias_24_prediction_error',
    'mae_24_bias_adjusted',
    'mape_24_bias_adjusted_pct',
    'mpe_24_bias_adjusted_pct',
    'bias_24_bias_adjusted',
    'mae_38_bias_adjusted',
    'mape_38_bias_adjusted_pct',
    'mpe_38_bias_adjusted_pct',
    'bias_38_bias_adjusted',
    'mape_holiday24_improvement_pct',
    'mape_window_bias_adjusted_pct',
    'mape_window_improvement_pct',
]].sort_values(['unique_id', 'target_date']).reset_index(drop=True)

In [ ]:
if 'bias_adjusted_profiles' not in globals() or not bias_adjusted_profiles:
    raise ValueError('Run Cell 10 first to build bias_adjusted_profiles before plotting raw vs adjusted forecasts.')
if 'rolling_daily_table' not in globals() or rolling_daily_table.empty:
    raise ValueError('Run Cell 10 first to enrich rolling_daily_table before plotting raw vs adjusted forecasts.')

batch_bias_plot_results = (
    batch_result_2025_all if 'batch_result_2025_all' in globals() and batch_result_2025_all else {}
)
if not batch_bias_plot_results:
    raise ValueError('No batch results are available to plot the raw vs adjusted forecasts.')

for series_unique_id, series_batch_result in batch_bias_plot_results.items():
    available_target_dates = [
        target_date
        for target_date, _ in series_batch_result.target_items
        if (series_unique_id, target_date) in bias_adjusted_profiles
    ]
    if not available_target_dates:
        continue

    ncols = 2
    nrows = int(np.ceil(len(available_target_dates) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(18, 4.8 * nrows), constrained_layout=True)
    axes = np.atleast_1d(axes).ravel()

    for axis, target_date in zip(axes, available_target_dates):
        profile_dict = bias_adjusted_profiles[(series_unique_id, target_date)]
        run = series_batch_result.runs[target_date]
        hours = analog_holidays_module._relative_hour_axis(run)
        tick_step = analog_holidays_module._hour_tick_step(run.season_length)
        metric_row = (
            rolling_daily_table
            .loc[(rolling_daily_table['unique_id'] == series_unique_id) & (rolling_daily_table['target_date'] == target_date)]
            .iloc[0]
        )
        metric_title_rows = analog_holidays_module._build_panel_metric_rows(metric_row)
        config_title_row = analog_holidays_module._build_panel_config_row(metric_row)
        axis.plot(hours, profile_dict['full_forecast'], color='#d62828', linestyle='--', linewidth=2.0, label='Raw forecast')
        axis.plot(hours, profile_dict['adjusted_full_forecast'], color='#1d3557', linewidth=2.3, label='Adjusted forecast')
        axis.plot(hours, profile_dict['full_actual'], color='#000000', linewidth=1.9, label='Actual')
        axis.axvline(0, color='#6c757d', linestyle=':', linewidth=1.0)
        axis.set_title(
            f"{target_date} | cluster={metric_row['analog_cluster']}\n"
            f"{config_title_row}\n"
            + "\n".join(metric_title_rows),
            fontsize=12,
        )
        axis.set_xlabel('Hour relative to holiday start | 38h window')
        axis.set_ylabel('Demand')
        axis.set_xticks(hours[::tick_step])
        axis.set_xlim(hours[0], hours[-1])
        axis.grid(alpha=0.2)
        axis.legend(fontsize=8)

    for axis in axes[len(available_target_dates):]:
        axis.set_visible(False)

    fig.suptitle(f'Raw vs adjusted forecast | 38h window | {series_unique_id}', y=1.02, fontsize=16)
    export_figure_pdf(fig, f'window38_raw_vs_adjusted_{series_unique_id}')
    plt.show()

In [ ]:
if 'rolling_daily_table' not in globals() or rolling_daily_table.empty:
    raise ValueError('Run Cell 10 first to build the bias-adjusted rolling_daily_table before the segmented error analysis.')
if 'mape_holiday24_bias_adjusted_pct' not in rolling_daily_table.columns:
    raise ValueError('Run Cell 10 first so the adjusted holiday24 KPI is available for the segmented error analysis.')

if 'selector_features_df' not in globals():
    selector_features_df = pd.read_csv(SELECTOR_FEATURES_PATH, parse_dates=['date'])
    selector_features_df['date'] = pd.to_datetime(selector_features_df['date']).dt.normalize()
if 'unique_id' not in selector_features_df.columns:
    raise ValueError(
        f'{SELECTOR_FEATURES_PATH.name} must contain a unique_id column. '
        'Re-export the selector from M_identify_holidays.ipynb.'
    )

selector_analysis_df = selector_features_df.copy().rename(
    columns={
        'analog_cluster': 'selector_analog_cluster',
        'day_class_code': 'selector_day_class_code',
        'daily_profile_cluster_id': 'daily_profile_cluster_id_raw',
        'event_profile_cluster_id': 'event_profile_cluster_id_raw',
    }
)
selector_analysis_df['unique_id'] = selector_analysis_df['unique_id'].astype(str)
selector_analysis_df['target_date'] = pd.to_datetime(selector_analysis_df['date']).dt.date.astype(str)
selector_analysis_df = selector_analysis_df.drop(columns=['date']).drop_duplicates(
    subset=['unique_id', 'target_date'],
    keep='last',
)

error_analysis_df = rolling_daily_table.copy()
error_analysis_df['unique_id'] = error_analysis_df['unique_id'].astype(str)
error_analysis_df['target_date'] = pd.to_datetime(error_analysis_df['target_date']).dt.date.astype(str)
error_analysis_df = error_analysis_df.merge(
    selector_analysis_df,
    on=['unique_id', 'target_date'],
    how='left',
    validate='many_to_one',
)

def _resolve_weekend_like_category(row):
    best_matching_weekday = row.get('best_matching_weekday')
    daily_profile_archetype = row.get('daily_profile_archetype')
    best_text = '' if pd.isna(best_matching_weekday) else str(best_matching_weekday).strip().lower()
    archetype_text = '' if pd.isna(daily_profile_archetype) else str(daily_profile_archetype).strip().lower()
    if best_text == 'saturday':
        return 'Saturday'
    if best_text == 'sunday':
        return 'Sunday'
    if 'saturday' in archetype_text:
        return 'Saturday-like'
    if 'sunday' in archetype_text:
        return 'Sunday-like'
    return 'Other / unclear'

error_analysis_df['weekend_like_category'] = error_analysis_df.apply(_resolve_weekend_like_category, axis=1)
error_analysis_df['k_label'] = error_analysis_df['k'].round().astype('Int64').astype(str)
error_analysis_df['scale_method_label'] = error_analysis_df['scale_method'].astype('string').fillna('None')
error_analysis_df['selected_analogs_label'] = error_analysis_df['selected_analogs'].round().astype('Int64').astype(str)
error_analysis_df['eligible_tuning_dates_label'] = error_analysis_df['eligible_tuning_dates'].round().astype('Int64').astype(str)
error_analysis_df['target_has_complete_profile_label'] = error_analysis_df['target_has_complete_profile'].map({True: 'complete', False: 'incomplete'})
error_analysis_df['fail_label'] = error_analysis_df['fail'].map({True: 'fail', False: 'ok'})
error_analysis_df['selector_day_class_code_label'] = error_analysis_df['selector_day_class_code'].astype('string')
error_analysis_df['daily_profile_cluster_id_label'] = error_analysis_df['daily_profile_cluster_id_raw'].astype('Int64').astype(str)
error_analysis_df['event_profile_cluster_id_label'] = error_analysis_df['event_profile_cluster_id_raw'].astype('Int64').astype(str)
error_analysis_df['is_fixed_date_label'] = error_analysis_df['is_fixed_date'].map({True: 'fixed', False: 'not_fixed'})
error_analysis_df['is_observed_monday_rule_label'] = error_analysis_df['is_observed_monday_rule'].map({True: 'observed_monday', False: 'not_observed_monday'})
PRIMARY_KPI = 'mape_holiday24_bias_adjusted_pct'
PRIMARY_KPI_LABEL = 'Holiday 24h bias-adjusted MAPE %'

model_segment_columns = [
    'unique_id',
    'typereg',
    'typedist',
    'k_label',
    'scale_method_label',
    'selected_analogs_label',
    'eligible_tuning_dates_label',
    'target_has_complete_profile_label',
    'fail_label',
]
selector_segment_columns = [
    'selector_analog_cluster',
    'holiday_name',
    'anchor_holiday_name',
    'holiday_day_type',
    'weekday_name',
    'selector_day_class_code_label',
    'day_class_name',
    'season',
    'date_rule',
    'is_fixed_date_label',
    'is_observed_monday_rule_label',
    'best_matching_weekday',
    'weekend_like_category',
    'daily_profile_cluster',
    'daily_profile_cluster_id_label',
    'daily_profile_archetype',
    'event_profile_cluster',
    'event_profile_cluster_id_label',
]
segment_columns = [
    column
    for column in model_segment_columns + selector_segment_columns
    if column in error_analysis_df.columns
]

def build_segment_metric_stats(df, feature, metric, min_count=2):
    working_df = df[[feature, metric]].copy().dropna()
    if working_df.empty:
        return pd.DataFrame()
    working_df[feature] = working_df[feature].astype(str)
    stats_df = (
        working_df.groupby(feature, dropna=False)[metric]
        .agg(
            count='size',
            mean='mean',
            median='median',
            q25=lambda values: values.quantile(0.25),
            q75=lambda values: values.quantile(0.75),
            max='max',
        )
        .reset_index()
    )
    stats_df = stats_df.loc[stats_df['count'] >= min_count].copy()
    if stats_df.empty:
        return stats_df
    return stats_df.sort_values(['median', 'mean', 'count'], ascending=[False, False, False]).reset_index(drop=True)

def build_feature_hotspots(df, features, metric='mape_holiday24_bias_adjusted_pct', min_count=2):
    hotspot_rows = []
    for feature in features:
        feature_stats_df = build_segment_metric_stats(df, feature, metric=metric, min_count=min_count)
        if feature_stats_df.empty:
            continue
        worst_row = feature_stats_df.iloc[0]
        hotspot_rows.append({
            'feature': feature,
            'worst_segment': worst_row[feature],
            'count': int(worst_row['count']),
            'mean': float(worst_row['mean']),
            'median': float(worst_row['median']),
            'q75': float(worst_row['q75']),
            'max': float(worst_row['max']),
        })
    if not hotspot_rows:
        return pd.DataFrame()
    return pd.DataFrame(hotspot_rows).sort_values(['median', 'q75', 'count'], ascending=[False, False, False]).reset_index(drop=True)

def build_all_segment_stats(df, features, metric='mape_holiday24_bias_adjusted_pct', min_count=2):
    stats_frames = []
    for feature in features:
        feature_stats_df = build_segment_metric_stats(df, feature, metric=metric, min_count=min_count)
        if feature_stats_df.empty:
            continue
        feature_stats_df = feature_stats_df.rename(columns={feature: 'segment_value'})
        feature_stats_df.insert(0, 'feature', feature)
        stats_frames.append(feature_stats_df)
    if not stats_frames:
        return pd.DataFrame()
    return pd.concat(stats_frames, ignore_index=True).sort_values(
        ['median', 'q75', 'count'],
        ascending=[False, False, False],
    ).reset_index(drop=True)

def _ordered_levels(stats_df, feature, max_levels):
    if feature in {
        'k_label',
        'selected_analogs_label',
        'eligible_tuning_dates_label',
        'selector_day_class_code_label',
        'daily_profile_cluster_id_label',
        'event_profile_cluster_id_label',
    }:
        filtered = stats_df.loc[stats_df[feature] != '<NA>', feature].tolist()
        filtered = sorted(filtered, key=lambda value: float(value))
        return filtered[:max_levels]
    return stats_df.head(max_levels)[feature].tolist()

def plot_segmented_error_boxplots(df, features, metric='mape_holiday24_bias_adjusted_pct', min_count=2, max_levels=10, ncols=2):
    valid_features = [feature for feature in features if feature in df.columns]
    if not valid_features:
        raise ValueError('No valid segment columns were found for the boxplot analysis.')

    nrows = int(np.ceil(len(valid_features) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(18, 4.8 * nrows), constrained_layout=True)
    axes = np.atleast_1d(axes).ravel()

    for axis, feature in zip(axes, valid_features):
        stats_df = build_segment_metric_stats(df, feature, metric=metric, min_count=min_count)
        if stats_df.empty:
            axis.set_visible(False)
            continue

        levels = _ordered_levels(stats_df, feature, max_levels=max_levels)
        plot_df = df.loc[df[feature].astype(str).isin(levels), [feature, metric]].copy().dropna()
        if plot_df.empty:
            axis.set_visible(False)
            continue

        plot_df[feature] = pd.Categorical(plot_df[feature].astype(str), categories=levels, ordered=True)
        plot_df.boxplot(column=metric, by=feature, ax=axis, rot=35, grid=False)
        counts = stats_df.set_index(feature).loc[levels, 'count'].astype(int).tolist()
        axis.set_title(feature)
        axis.set_xlabel('')
        axis.set_ylabel(metric)
        axis.set_xticklabels([f'{level}\n(n={count})' for level, count in zip(levels, counts)], rotation=35, ha='right')

    for axis in axes[len(valid_features):]:
        axis.set_visible(False)

    fig.suptitle(f'Error boxplots by segment | metric={metric}', y=1.01, fontsize=14)
    return fig, axes

analysis_min_count = 2
boxplot_max_levels = 10

mape_feature_hotspots_df = build_feature_hotspots(
    error_analysis_df,
    segment_columns,
    metric=PRIMARY_KPI,
    min_count=analysis_min_count,
    )
mae_feature_hotspots_df = build_feature_hotspots(
    error_analysis_df,
    segment_columns,
    metric='mae_holiday24_bias_adjusted',
    min_count=analysis_min_count,
    )
segment_mape_stats_df = build_all_segment_stats(
    error_analysis_df,
    segment_columns,
    metric=PRIMARY_KPI,
    min_count=analysis_min_count,
    )

print(
    f'Rows analyzed: {len(error_analysis_df)} | series: {error_analysis_df["unique_id"].nunique()} | '
    f'target dates: {error_analysis_df["target_date"].nunique()}'
)
display(
    error_analysis_df[[
        'unique_id',
        'target_date',
        'holiday_label',
        'selector_analog_cluster',
        'holiday_day_type',
        'typereg',
        'typedist',
        'k',
        'scale_method_label',
        'weekend_like_category',
        'mae_holiday24_bias_adjusted',
        'mape_holiday24_raw_pct',
        'mape_holiday24_bias_adjusted_pct',
        'mape_holiday24_improvement_pct',
    ]].head(12)
)

print(f'Worst segment by feature | {PRIMARY_KPI_LABEL}')
display(mape_feature_hotspots_df.head(20))

print('Worst segment by feature | MAE')
display(mae_feature_hotspots_df.head(20))

print(f'All segment stats ranked by median {PRIMARY_KPI_LABEL}')
display(segment_mape_stats_df.head(50))

fig_model_segments, _ = plot_segmented_error_boxplots(
    error_analysis_df,
    model_segment_columns,
    metric=PRIMARY_KPI,
    min_count=analysis_min_count,
    max_levels=boxplot_max_levels,
    )
plt.show()

fig_selector_segments, _ = plot_segmented_error_boxplots(
    error_analysis_df,
    selector_segment_columns,
    metric=PRIMARY_KPI,
    min_count=analysis_min_count,
    max_levels=boxplot_max_levels,
    )
plt.show()

In [ ]:
if 'rolling_runs' not in globals() or not rolling_runs:
    raise ValueError('Run Cell 9 first to build rolling_runs before the OOS action-priority study.')

if 'rolling_daily_table' not in globals() or rolling_daily_table.empty:
    raise ValueError('Run Cell 9 first to build rolling_daily_table before the OOS action-priority study.')

if 'selector_features_df' not in globals():
    selector_features_df = pd.read_csv(SELECTOR_FEATURES_PATH, parse_dates=['date'])
    selector_features_df['date'] = pd.to_datetime(selector_features_df['date']).dt.normalize()
if 'unique_id' not in selector_features_df.columns:
    raise ValueError(
        f'{SELECTOR_FEATURES_PATH.name} must contain a unique_id column. '
        'Re-export the selector from M_identify_holidays.ipynb.'
    )

selector_priority_df = selector_features_df.copy().rename(
    columns={
        'analog_cluster': 'selector_analog_cluster',
        'day_class_code': 'selector_day_class_code',
        'daily_profile_cluster_id': 'daily_profile_cluster_id_raw',
        'event_profile_cluster_id': 'event_profile_cluster_id_raw',
    }
)
selector_priority_df['unique_id'] = selector_priority_df['unique_id'].astype(str)
selector_priority_df['target_date'] = pd.to_datetime(selector_priority_df['date']).dt.date.astype(str)
selector_priority_df = selector_priority_df.drop(columns=['date']).drop_duplicates(
    subset=['unique_id', 'target_date'],
    keep='last',
)

def _resolve_weekend_like_category_priority(row):
    best_matching_weekday = row.get('best_matching_weekday')
    daily_profile_archetype = row.get('daily_profile_archetype')
    best_text = '' if pd.isna(best_matching_weekday) else str(best_matching_weekday).strip().lower()
    archetype_text = '' if pd.isna(daily_profile_archetype) else str(daily_profile_archetype).strip().lower()
    if best_text == 'saturday':
        return 'Saturday'
    if best_text == 'sunday':
        return 'Sunday'
    if 'saturday' in archetype_text:
        return 'Saturday-like'
    if 'sunday' in archetype_text:
        return 'Sunday-like'
    return 'Other / unclear'

if 'oos_point_error_df' not in globals() or oos_point_error_df.empty:
    oos_priority_rows = []
    for unique_id, series_run_map in rolling_runs.items():
        for target_date, run in series_run_map.items():
            if run is None or run.actual_profile is None:
                continue

            target_ts = pd.Timestamp(target_date).normalize()
            forecast_values = np.asarray(run.forecast_profile, dtype=float)
            actual_values = np.asarray(run.actual_profile, dtype=float)
            horizon = min(forecast_values.size, actual_values.size)
            recent_weekend_dates = getattr(run, 'recent_weekend_dates', []) or []
            recent_weekend_like = getattr(run, 'recent_weekend_like', None)
            recent_weekend_like = recent_weekend_like if recent_weekend_like is not None else 'None'

            for hour_idx in range(horizon):
                forecast_timestamp = run.forecast_start + pd.Timedelta(hours=hour_idx)
                hour_relative = hour_idx - int(run.forecast_start_offset_hours)
                actual_value = float(actual_values[hour_idx])
                forecast_value = float(forecast_values[hour_idx])
                signed_error = forecast_value - actual_value
                abs_error = abs(signed_error)
                ape_pct = np.nan
                if abs(actual_value) > 1e-9:
                    ape_pct = abs_error / abs(actual_value) * 100.0

                oos_priority_rows.append({
                    'unique_id': unique_id,
                    'target_date': target_ts.date().isoformat(),
                    'forecast_timestamp': forecast_timestamp,
                    'forecast_hour_of_day': int(forecast_timestamp.hour),
                    'window_slice': 'pre_holiday_14h' if hour_relative < 0 else 'holiday_24h',
                    'hour_relative_to_holiday': hour_relative,
                    'forecast_value': forecast_value,
                    'actual_value': actual_value,
                    'signed_error_oos': signed_error,
                    'abs_error_oos': abs_error,
                    'ape_pct_oos': ape_pct,
                    'recent_weekend_like_run': str(recent_weekend_like),
                    'recent_weekend_analogs_added': len(recent_weekend_dates),
                })

    if not oos_priority_rows:
        raise ValueError('No OOS forecast points with actuals were found in rolling_runs.')

    oos_point_error_df = pd.DataFrame(oos_priority_rows)
    oos_point_error_df = oos_point_error_df.merge(
        rolling_daily_table,
        on=['unique_id', 'target_date'],
        how='left',
        validate='many_to_one',
    )
    oos_point_error_df = oos_point_error_df.merge(
        selector_priority_df,
        on=['unique_id', 'target_date'],
        how='left',
        validate='many_to_one',
    )

oos_point_error_df['unique_id'] = oos_point_error_df['unique_id'].astype(str)
oos_point_error_df['weekend_like_category'] = oos_point_error_df.apply(_resolve_weekend_like_category_priority, axis=1)
oos_point_error_df['k_label'] = oos_point_error_df['k'].round().astype('Int64').astype(str)
oos_point_error_df['scale_method_label'] = oos_point_error_df['scale_method'].astype('string').fillna('None')
oos_point_error_df['selected_analogs_label'] = oos_point_error_df['selected_analogs'].round().astype('Int64').astype(str)
oos_point_error_df['eligible_tuning_dates_label'] = oos_point_error_df['eligible_tuning_dates'].round().astype('Int64').astype(str)
oos_point_error_df['hour_relative_to_holiday_label'] = oos_point_error_df['hour_relative_to_holiday'].astype('Int64').astype(str)
oos_point_error_df['recent_weekend_analogs_added_label'] = oos_point_error_df['recent_weekend_analogs_added'].astype('Int64').astype(str)

priority_segment_columns = [
    'unique_id',
    'selector_analog_cluster',
    'holiday_name',
    'anchor_holiday_name',
    'holiday_day_type',
    'weekday_name',
    'season',
    'date_rule',
    'best_matching_weekday',
    'weekend_like_category',
    'daily_profile_archetype',
    'event_profile_cluster',
    'typereg',
    'typedist',
    'k_label',
    'scale_method_label',
    'selected_analogs_label',
    'eligible_tuning_dates_label',
    'window_slice',
    'hour_relative_to_holiday_label',
    'recent_weekend_like_run',
    'recent_weekend_analogs_added_label',
]
priority_segment_columns = [
    column for column in priority_segment_columns if column in oos_point_error_df.columns
]

def build_high_error_low_sample_table(df, features, min_hour_samples=8, min_date_samples=2):
    summary_rows = []
    for feature in features:
        working_df = df[[
            feature,
            'target_date',
            'ape_pct_oos',
            'abs_error_oos',
            'selected_analogs',
            'eligible_tuning_dates',
            'recent_weekend_analogs_added',
        ]].copy()
        working_df = working_df.dropna(subset=[feature, 'ape_pct_oos', 'abs_error_oos'])
        if working_df.empty:
            continue

        feature_summary_df = (
            working_df.groupby(feature, dropna=False)
            .agg(
                hour_samples=('ape_pct_oos', 'size'),
                date_samples=('target_date', 'nunique'),
                mean_ape_pct=('ape_pct_oos', 'mean'),
                median_ape_pct=('ape_pct_oos', 'median'),
                q75_ape_pct=('ape_pct_oos', lambda values: values.quantile(0.75)),
                max_ape_pct=('ape_pct_oos', 'max'),
                mean_abs_error=('abs_error_oos', 'mean'),
                median_abs_error=('abs_error_oos', 'median'),
                median_selected_analogs=('selected_analogs', 'median'),
                median_eligible_tuning_dates=('eligible_tuning_dates', 'median'),
                median_recent_weekend_analogs_added=('recent_weekend_analogs_added', 'median'),
            )
            .reset_index()
        )
        feature_summary_df = feature_summary_df.loc[
            (feature_summary_df['hour_samples'] >= min_hour_samples)
            & (feature_summary_df['date_samples'] >= min_date_samples)
        ].copy()
        if feature_summary_df.empty:
            continue

        feature_summary_df = feature_summary_df.rename(columns={feature: 'segment_value'})
        feature_summary_df.insert(0, 'feature', feature)
        summary_rows.append(feature_summary_df)

    if not summary_rows:
        return pd.DataFrame()

    priority_df = pd.concat(summary_rows, ignore_index=True)
    priority_df['error_pressure_score'] = (
        0.6 * priority_df['median_ape_pct'].rank(pct=True)
        + 0.4 * priority_df['q75_ape_pct'].rank(pct=True)
    )
    priority_df['sample_scarcity_score'] = (
        0.6 * (1.0 - priority_df['hour_samples'].rank(pct=True))
        + 0.4 * (1.0 - priority_df['date_samples'].rank(pct=True))
    )
    priority_df['analog_scarcity_score'] = (
        0.5 * (1.0 - priority_df['median_selected_analogs'].rank(pct=True))
        + 0.5 * (1.0 - priority_df['median_eligible_tuning_dates'].rank(pct=True))
    )
    priority_df['priority_score'] = 100.0 * (
        0.55 * priority_df['error_pressure_score']
        + 0.30 * priority_df['sample_scarcity_score']
        + 0.15 * priority_df['analog_scarcity_score']
    )
    return priority_df.sort_values(
        ['priority_score', 'median_ape_pct', 'sample_scarcity_score'],
        ascending=[False, False, False],
    ).reset_index(drop=True)

def _recommend_priority_action(row):
    feature = str(row['feature'])
    median_selected_analogs = float(row['median_selected_analogs'])
    median_eligible_tuning_dates = float(row['median_eligible_tuning_dates'])
    weekend_analogs_added = float(row['median_recent_weekend_analogs_added'])

    if feature in {'selector_analog_cluster', 'event_profile_cluster', 'holiday_day_type', 'holiday_name', 'anchor_holiday_name'}:
        return 'Relajar filtro de cluster/subtipo o fusionar segmentos sparsos.'
    if feature in {'weekend_like_category', 'best_matching_weekday', 'daily_profile_archetype', 'recent_weekend_like_run'} and weekend_analogs_added < float(RECENT_WEEKEND_ANALOGS):
        return 'Agregar mas analogs recientes tipo sabado/domingo antes del ranking.'
    if feature in {'selected_analogs_label', 'eligible_tuning_dates_label', 'k_label'} or median_selected_analogs <= 3.5 or median_eligible_tuning_dates <= 4.5:
        return 'Ampliar el pool candidato antes del ranking o relajar filtros de seleccion.'
    if feature in {'window_slice', 'hour_relative_to_holiday_label'}:
        return 'Separar reglas para pre-holiday vs holiday o ajustar por bloque horario.'
    return 'Inspeccionar el segmento y considerar relajar filtros o sumar mas ejemplos recientes.'

def _priority_reason(row):
    return (
        f"APE mediana={row['median_ape_pct']:.2f}% | q75={row['q75_ape_pct']:.2f}% | "
        f"horas={int(row['hour_samples'])} | fechas={int(row['date_samples'])} | "
        f"analogs mediana={row['median_selected_analogs']:.1f} | elegibles medianos={row['median_eligible_tuning_dates']:.1f}"
    )

oos_action_priority_df = build_high_error_low_sample_table(
    oos_point_error_df,
    priority_segment_columns,
    min_hour_samples=8,
    min_date_samples=2,
 )

if oos_action_priority_df.empty:
    raise ValueError('No segment reached the minimum OOS sample thresholds for the priority table.')

oos_action_priority_df['priority_band'] = pd.cut(
    oos_action_priority_df['priority_score'],
    bins=[-np.inf, 45, 60, 75, np.inf],
    labels=['Monitor', 'Medium', 'High', 'Critical'],
)
oos_action_priority_df['recommended_action'] = oos_action_priority_df.apply(_recommend_priority_action, axis=1)
oos_action_priority_df['reason'] = oos_action_priority_df.apply(_priority_reason, axis=1)

critical_oos_action_priority_df = oos_action_priority_df.loc[
    oos_action_priority_df['priority_band'].isin(['High', 'Critical'])
] .copy()

print('Automatic OOS priority table | high error + low sample')
display(
    oos_action_priority_df[[
        'feature',
        'segment_value',
        'priority_band',
        'priority_score',
        'hour_samples',
        'date_samples',
        'median_ape_pct',
        'q75_ape_pct',
        'median_abs_error',
        'median_selected_analogs',
        'median_eligible_tuning_dates',
        'median_recent_weekend_analogs_added',
        'recommended_action',
        'reason',
    ]].head(40)
)

print('Critical / high-priority OOS segments to inspect first')
display(
    critical_oos_action_priority_df[[
        'feature',
        'segment_value',
        'priority_band',
        'priority_score',
        'hour_samples',
        'date_samples',
        'median_ape_pct',
        'q75_ape_pct',
        'median_abs_error',
        'recommended_action',
        'reason',
    ]].head(25)
)

In [ ]:
import importlib.util

current_series_optuna_results = rolling_optuna_results.get(UNIQUE_ID, {}) if 'rolling_optuna_results' in globals() else {}
current_rolling_optuna_result = current_series_optuna_results.get(TARGET_DATE)
current_cluster_row = (
    rolling_daily_table.loc[
        (rolling_daily_table['unique_id'] == UNIQUE_ID)
        & (rolling_daily_table['target_date'] == TARGET_DATE)
    ]
    if 'rolling_daily_table' in globals() else pd.DataFrame()
)
current_cluster = current_cluster_row['analog_cluster'].iloc[0] if not current_cluster_row.empty else pd.NA
current_regressor_params = (
    dict(current_rolling_optuna_result.best_config.get('regressor_params', {}))
    if current_rolling_optuna_result is not None else {}
)
current_k_range = (
    tuple(current_rolling_optuna_result.best_config.get('k_range', (np.nan, np.nan)))
    if current_rolling_optuna_result is not None else (np.nan, np.nan)
)
current_scale_method = (
    current_rolling_optuna_result.best_config.get('scale_method', SCALE_METHOD)
    if current_rolling_optuna_result is not None else SCALE_METHOD
)

optuna_typedist_choices = ['pearson', 'euclidian']
optuna_typereg_choices = ['PCR', 'PLS']
lgbm_available = importlib.util.find_spec('lightgbm') is not None

optuna_k_rule = f'integer in [{current_k_range[0]}, {current_k_range[1]}]'
optuna_scale_method_rule = OPTUNA_SCALE_METHOD_CHOICES if OPTUNA_SCALE_METHOD_CHOICES is not None else [SCALE_METHOD]
optuna_n_components_rule = 'integer in [2, min(k, season_length)] only when typereg is PCR or PLS'
optuna_lgbm_rule = (
    'available only via explicit typereg_choices override: n_estimators in {100, 200, 300}, '
    'learning_rate in {0.03, 0.05, 0.1}, num_leaves in {15, 31, 63}, min_child_samples in {10, 20, 30}'
)
optuna_runtime_note = (
    'DTW, RidgeReg, LassoReg, RF, OLSstep, and LGBM are excluded from the default Optuna grid, '
    'and k is capped by the realizable post-filter analog pool so Optuna does not request '
    'more neighbors than the workflow can actually use.'
)

if current_rolling_optuna_result is None:
    rolling_daily_table.loc[
        (rolling_daily_table['unique_id'] == UNIQUE_ID)
        & (rolling_daily_table['target_date'] == TARGET_DATE)
    ]
else:
    optuna_method_report = (
        f'OPTUNA METHOD REPORT FOR UNIQUE_ID={UNIQUE_ID} | TARGET_DATE={TARGET_DATE}\n'
        f'- Optimization mode: single-objective minimization.\n'
        f'- Sampler: TPESampler(seed={OPTUNA_RANDOM_SEED}).\n'
        f'- Search budget: n_trials={OPTUNA_N_TRIALS}, timeout_sec={OPTUNA_TIMEOUT_SEC}.\n'
        f'- Rolling cutoff: train_end={TARGET_DATE}; only dates strictly earlier than the target are used for tuning.\n'
        f'- Backtest folds: {len(current_rolling_optuna_result.eligible_dates)} eligible historical holiday dates, capped by OPTUNA_MAX_EVAL_DATES={OPTUNA_MAX_EVAL_DATES}.\n'
        f'- Cluster restriction: match_target_cluster={MATCH_TARGET_CLUSTER}; target analog_cluster={current_cluster if MATCH_TARGET_CLUSTER else "disabled"}.\n'
        f'- Objective function: minimize mean MAE across historical folds + 1000 * fail_rate.\n'
        f'- Search space for typedist: {optuna_typedist_choices}.\n'
        f'- Search space for typereg: {optuna_typereg_choices}.\n'
        f'- Search space for scale_method: {optuna_scale_method_rule}.\n'
        f'- Search space for k: {optuna_k_rule}.\n'
        f'- Conditional parameter for n_components: {optuna_n_components_rule}.\n'
        f'- Conditional LGBM parameters: '
        f'{optuna_lgbm_rule if lgbm_available else "not available because lightgbm is not installed"}.\n'
        f'- Runtime note: {optuna_runtime_note}\n'
        f'- Best configuration selected for this target: k(optuna)={current_rolling_optuna_result.best_config["k"]}, '
        f'typedist(optuna)={current_rolling_optuna_result.best_config["typedist"]}, '
        f'typereg(optuna)={current_rolling_optuna_result.best_config["typereg"]}, '
        f'scale_method(optuna)={current_scale_method}, '
        f'n_components(optuna)={current_rolling_optuna_result.best_config["n_components"]}, '
        f'regressor_params(optuna)={current_regressor_params}.\n'
        f'- k-range(optuna-param)=[{current_k_range[0]}, {current_k_range[1]}].'
    )
    print(optuna_method_report)
    display(current_rolling_optuna_result.summary_df)
    current_rolling_optuna_result.fold_metrics_df

In [ ]:
if (
    ('batch_result_2025_all' not in globals() or not batch_result_2025_all)
    and 'rolling_runs' in globals()
    and 'rolling_daily_table' in globals()
    and not rolling_daily_table.empty
    and 'rolling_target_items' in globals()
):
    batch_result_2025_all = {}
    for series_unique_id, series_run_map in rolling_runs.items():
        series_results_df = (
            rolling_daily_table
            .loc[rolling_daily_table['unique_id'] == series_unique_id]
            .reset_index(drop=True)
        )
        if series_results_df.empty:
            continue
        metric_summary_columns = [
            column for column in [
                'mae_holiday24_bias_adjusted',
                'mape_holiday24_bias_adjusted_pct',
                'mae_holiday24_raw',
                'mape_holiday24_raw_pct',
                'mae_window_bias_adjusted',
                'mape_window_bias_adjusted_pct',
                'mae_window',
                'mape_window_pct',
            ]
            if column in series_results_df.columns
        ]
        batch_result_2025_all[series_unique_id] = analog_holidays_module.AnalogHolidayBatchResult(
            target_items=rolling_target_items,
            runs=series_run_map,
            results_df=series_results_df,
            metric_summary_df=series_results_df[metric_summary_columns].describe(include='all'),
        )

batch_result_2025 = batch_result_2025_all.get(UNIQUE_ID) if 'batch_result_2025_all' in globals() else None
batch_results_to_plot = (
    batch_result_2025_all if 'batch_result_2025_all' in globals() and batch_result_2025_all else {}
)
if not batch_results_to_plot:
    raise ValueError('Run Cell 9 first to build rolling batch results before plotting.')

batch_inference_figures = {}
batch_inference_axes = {}

for series_unique_id, series_batch_result in batch_results_to_plot.items():
    print(f'Batch inference grid | {series_unique_id}')
    fig, axes = plot_batch_inference_grid(
        series_batch_result,
        title=(
            f'Batch inference | {series_unique_id}\n'
            f'Rolling nested tuning by target date | '
            f'window={SEASON_LENGTH}h | start=-{FORECAST_START_OFFSET_HOURS}h'
        ),
    )
    batch_inference_figures[series_unique_id] = fig
    batch_inference_axes[series_unique_id] = axes
    export_figure_pdf(fig, f'batch_inference_{series_unique_id}')
    plt.show()

In [ ]:
current_series_optuna_results = rolling_optuna_results.get(UNIQUE_ID, {}) if 'rolling_optuna_results' in globals() else {}
current_rolling_optuna_result = current_series_optuna_results.get(TARGET_DATE)
resolved_regressor_params = dict(globals().get('REGRESSOR_PARAMS', {}))
resolved_scale_method = globals().get('SCALE_METHOD')
if current_rolling_optuna_result is not None:
    optuna_result = current_rolling_optuna_result
    K = int(optuna_result.best_config['k'])
    TYPEDIST = str(optuna_result.best_config['typedist'])
    TYPEREG = str(optuna_result.best_config['typereg'])
    SCALE_METHOD = optuna_result.best_config.get('scale_method', SCALE_METHOD)
    N_COMPONENTS = int(optuna_result.best_config['n_components'])
    resolved_regressor_params = dict(optuna_result.best_config.get('regressor_params', {}))
    resolved_scale_method = SCALE_METHOD
    REGRESSOR_PARAMS = resolved_regressor_params

SOURCE_PATH = _ensure_working_source_path(SOURCE_PATH)
run = batch_result_2025.runs.get(TARGET_DATE) if 'batch_result_2025' in globals() and batch_result_2025 is not None else None

if run is None:
    run = run_analog_holidays(
        unique_id=UNIQUE_ID,
        target_date=TARGET_DATE,
        source_path=SOURCE_PATH,
        season_length=SEASON_LENGTH,
        forecast_start_offset_hours=FORECAST_START_OFFSET_HOURS,
        k=K,
        typedist=TYPEDIST,
        typereg=TYPEREG,
        scale_method=resolved_scale_method,
        n_components=N_COMPONENTS,
        regressor_params=resolved_regressor_params,
        levels=LEVELS,
        special_labels=SPECIAL_LABELS,
        min_special_points=MIN_SPECIAL_POINTS,
        min_event_gap=MIN_EVENT_GAP,
        max_events=MAX_EVENTS,
        selector_features_path=SELECTOR_FEATURES_PATH,
        cluster_column=CLUSTER_COLUMN,
        match_target_cluster=MATCH_TARGET_CLUSTER,
        recent_weekend_analogs=RECENT_WEEKEND_ANALOGS,
    )

summary_df = build_run_summary(run)
print(summary_df.to_string(index=False))

In [ ]:
current_series_optuna_results = rolling_optuna_results.get(UNIQUE_ID, {}) if 'rolling_optuna_results' in globals() else {}
current_rolling_optuna_result = current_series_optuna_results.get(TARGET_DATE)
resolved_regressor_params = dict(globals().get('REGRESSOR_PARAMS', {}))
resolved_scale_method = globals().get('SCALE_METHOD')
if current_rolling_optuna_result is not None:
    optuna_result = current_rolling_optuna_result
    K = int(optuna_result.best_config['k'])
    TYPEDIST = str(optuna_result.best_config['typedist'])
    TYPEREG = str(optuna_result.best_config['typereg'])
    SCALE_METHOD = optuna_result.best_config.get('scale_method', SCALE_METHOD)
    N_COMPONENTS = int(optuna_result.best_config['n_components'])
    resolved_regressor_params = dict(optuna_result.best_config.get('regressor_params', {}))
    resolved_scale_method = SCALE_METHOD
    REGRESSOR_PARAMS = resolved_regressor_params

SOURCE_PATH = _ensure_working_source_path(SOURCE_PATH)
diagnostic_run = batch_result_2025.runs.get(TARGET_DATE) if 'batch_result_2025' in globals() and batch_result_2025 is not None else None

if diagnostic_run is None:
    diagnostic_run = run_analog_holidays(
        unique_id=UNIQUE_ID,
        target_date=TARGET_DATE,
        source_path=SOURCE_PATH,
        season_length=SEASON_LENGTH,
        forecast_start_offset_hours=FORECAST_START_OFFSET_HOURS,
        k=K,
        typedist=TYPEDIST,
        typereg=TYPEREG,
        scale_method=resolved_scale_method,
        n_components=N_COMPONENTS,
        regressor_params=resolved_regressor_params,
        levels=LEVELS,
        special_labels=SPECIAL_LABELS,
        min_special_points=MIN_SPECIAL_POINTS,
        min_event_gap=MIN_EVENT_GAP,
        max_events=MAX_EVENTS,
        expected_target_label=None,
        selector_features_path=SELECTOR_FEATURES_PATH,
        cluster_column=CLUSTER_COLUMN,
        match_target_cluster=MATCH_TARGET_CLUSTER,
        recent_weekend_analogs=RECENT_WEEKEND_ANALOGS,
    )

interval_rows = []
for lv in LEVELS:
    lo = diagnostic_run.interval_low.get(lv)
    hi = diagnostic_run.interval_high.get(lv)
    if lo is None or hi is None:
        continue
    width = hi - lo
    interval_rows.append({
        'level': lv,
        'average_interval_width': float(np.mean(width)),
        'max_interval_width': float(np.max(width)),
        'min_interval_width': float(np.min(width)),
    })

display(pd.DataFrame(interval_rows))

level_to_show = 95 if 95 in LEVELS else max(LEVELS)
hourly_interval_df = pd.DataFrame({
    'hour_relative_to_holiday': np.arange(len(diagnostic_run.forecast_profile)) - FORECAST_START_OFFSET_HOURS,
    'forecast_mean': diagnostic_run.forecast_profile,
    f'lower_limit_{level_to_show}': diagnostic_run.interval_low[level_to_show],
    f'upper_limit_{level_to_show}': diagnostic_run.interval_high[level_to_show],
})

hourly_interval_df.head(10)

### X/X' and Y/Y' Sequence Batch Grid

One chart per forecast date, showing the historical X/X' pairs in light blue and the Y/Y' forecast sequence in red.

In this variant the X'/Y' window spans 38 hours and starts 14 hours before the holiday begins.

In [ ]:
batch_results_to_plot = (
    batch_result_2025_all if 'batch_result_2025_all' in globals() and batch_result_2025_all else {UNIQUE_ID: batch_result_2025}
    if 'batch_result_2025' in globals() and batch_result_2025 is not None else {}
 )
if not batch_results_to_plot:
    raise ValueError('No batch results are available to plot.')

batch_pair_figures = {}
batch_pair_axes = {}

for series_unique_id, series_batch_result in batch_results_to_plot.items():
    print(f"X/X' and Y/Y' grid | {series_unique_id}")
    adjusted_forecasts_by_date = None
    if 'bias_adjusted_profiles' in globals() and bias_adjusted_profiles:
        adjusted_forecasts_by_date = {
            target_date: bias_adjusted_profiles[(series_unique_id, target_date)]['adjusted_full_forecast']
            for target_date, _ in series_batch_result.target_items
            if (series_unique_id, target_date) in bias_adjusted_profiles
        }
    fig_seq, axes_seq = plot_batch_pair_sequences_grid(
        series_batch_result,
        title=(
            f"X/X' and Y/Y' by forecast date | {series_unique_id}\n"
            f"Rolling nested tuning by target date | "
            f"window={SEASON_LENGTH}h | start=-{FORECAST_START_OFFSET_HOURS}h"
        ),
        adjusted_forecasts_by_date=adjusted_forecasts_by_date,
    )
    batch_pair_figures[series_unique_id] = fig_seq
    batch_pair_axes[series_unique_id] = axes_seq
    export_figure_pdf(fig_seq, f'batch_pair_sequences_{series_unique_id}')
    plt.show()